# Grade 11 · Unit 6 — Time series and seasonality

> **SYNTHETIC** monthly climate table.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")

clim = pd.read_csv(data_file("climate-monthly", "climate_monthly_synthetic.csv"))
print(clim.shape)

## Seasonal index: how far is each month from the yearly average?

In [ ]:
clim["t"] = clim["year"] + (clim["month"] - 1) / 12
month_mean = clim.groupby("month")["mean_temp_c"].mean()
seasonal = month_mean - month_mean.mean()
clim["deseason"] = clim["mean_temp_c"] - clim["month"].map(seasonal)
assert abs(seasonal.sum()) < 1e-9
seasonal.round(1)

## Trend on the de-seasonalised series

In [ ]:
slope, intercept = np.polyfit(clim["t"], clim["deseason"], 1)
print(f"trend ≈ {10*slope:.2f} °C per decade")
plt.plot(clim["t"], clim["deseason"], lw=0.7); plt.plot(clim["t"], intercept + slope * clim["t"], color="red")
plt.ylabel("de-seasonalised °C"); plt.show()

## Uncertainty: bootstrap over years
**Predict:** will the interval include 0?

In [ ]:
rng = np.random.default_rng(3)
years = clim["year"].unique()
slopes = []
for _ in range(1000):
    pick = rng.choice(years, size=len(years), replace=True)
    d = pd.concat([clim[clim["year"] == y] for y in pick])
    slopes.append(10 * np.polyfit(d["t"], d["deseason"], 1)[0])
lo, hi = np.percentile(slopes, [2.5, 97.5])
assert lo < 10 * slope < hi
print(f"95% bootstrap interval for the trend: {lo:.2f} to {hi:.2f} °C per decade")

**Make:** write a sentence on the trend (with its interval) and one on what the data cannot tell us.